In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

***Fetching the Datasets***

In [2]:
train_dataset = pd.read_csv("data/train.csv")
test_dataset = pd.read_csv("data/test.csv")

# Drop the target and non-predictive columns for X
X_train = train_dataset.drop(['Survived', 'PassengerId', 'Name', 'Ticket'], axis=1)

# Only keep the target for y
y_train = train_dataset['Survived']

# Drop the target and non-predictive columns for X
X_test = test_dataset.drop(['PassengerId', 'Name', 'Ticket'], axis=1)

***Analysing and Visualizing the Datasets***

In [3]:
print(f"\n{X_train.info()}")
print(f"\n{X_test.info()}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    891 non-null    int64  
 1   Sex       891 non-null    object 
 2   Age       714 non-null    float64
 3   SibSp     891 non-null    int64  
 4   Parch     891 non-null    int64  
 5   Fare      891 non-null    float64
 6   Cabin     204 non-null    object 
 7   Embarked  889 non-null    object 
dtypes: float64(2), int64(3), object(3)
memory usage: 55.8+ KB

None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    418 non-null    int64  
 1   Sex       418 non-null    object 
 2   Age       332 non-null    float64
 3   SibSp     418 non-null    int64  
 4   Parch     418 non-null    int64  
 5   Fare      417 non-null    float64
 6   Cabin     91 non

In [4]:
print(f"Missing values in X_train:\n{X_train.isna().sum()}")
X_train.describe(include='all')

Missing values in X_train:
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Cabin       687
Embarked      2
dtype: int64


,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked
count,891.000000,891,714.000000,891.000000,891.000000,891.000000,204,889
unique,NaN,2,NaN,NaN,NaN,NaN,147,3
top,NaN,male,NaN,NaN,NaN,NaN,B96 B98,S
freq,NaN,577,NaN,NaN,NaN,NaN,4,644
mean,2.308642,NaN,29.699118,0.523008,0.381594,32.204208,NaN,NaN
std,0.836071,NaN,14.526497,1.102743,0.806057,49.693429,NaN,NaN
min,1.000000,NaN,0.420000,0.000000,0.000000,0.000000,NaN,NaN
25%,2.000000,NaN,20.125000,0.000000,0.000000,7.910400,NaN,NaN
50%,3.000000,NaN,28.000000,0.000000,0.000000,14.454200,NaN,NaN
75%,3.000000,NaN,38.000000,1.000000,0.000000,31.000000,NaN,NaN


In [5]:
print(f"Missing values in X_test:\n{X_test.isna().sum()}")
X_test.describe(include='all')

Missing values in X_test:
Pclass        0
Sex           0
Age          86
SibSp         0
Parch         0
Fare          1
Cabin       327
Embarked      0
dtype: int64


,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked
count,418.000000,418,332.000000,418.000000,418.000000,417.000000,91,418
unique,NaN,2,NaN,NaN,NaN,NaN,76,3
top,NaN,male,NaN,NaN,NaN,NaN,B57 B59 B63 B66,S
freq,NaN,266,NaN,NaN,NaN,NaN,3,270
mean,2.265550,NaN,30.272590,0.447368,0.392344,35.627188,NaN,NaN
std,0.841838,NaN,14.181209,0.896760,0.981429,55.907576,NaN,NaN
min,1.000000,NaN,0.170000,0.000000,0.000000,0.000000,NaN,NaN
25%,1.000000,NaN,21.000000,0.000000,0.000000,7.895800,NaN,NaN
50%,3.000000,NaN,27.000000,0.000000,0.000000,14.454200,NaN,NaN
75%,3.000000,NaN,39.000000,1.000000,0.000000,31.500000,NaN,NaN


In [6]:
# Fill Age based on the average age of that specific Pclass
X_train["Age"] = X_train["Age"].fillna(
    X_train.groupby("Pclass")["Age"].transform("mean")
)

# Create a new category for missing cabins
X_train['Cabin'] = X_train['Cabin'].fillna('Unknown')

# PRO TIP: Just take the first letter of the Cabin (The Deck)
# Decks A, B, C were for wealthier passengers.
X_train['Deck'] = X_train['Cabin'].str[0]

# This automatically turns all text columns into 1s and 0s
X_train = pd.get_dummies(X_train, columns=['Embarked', 'Sex'])

# Method A: Manual Mapping (Best for complete control)
deck_mapping = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7, "T": 8, "U": 8}
X_train['Deck_Encoded'] = X_train['Deck'].map(deck_mapping)

'''
One Tiny "Gotcha" to Check
There is one small thing that might trip you up later: The "Unknown" Deck.

When you ran train_dataset['Cabin'].str[0] on the values you filled with "Unknown",
your new Deck column now contains the letter 'U' (the first letter of "Unknown").

This is actually good! It gives your model a specific category for "People with 
no recorded cabin."

Just be careful: Ensure your final model treats 'U' as a category and not as an 
actual deck near 'T' or 'V'.
'''
print(X_train.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Pclass        891 non-null    int64  
 1   Age           891 non-null    float64
 2   SibSp         891 non-null    int64  
 3   Parch         891 non-null    int64  
 4   Fare          891 non-null    float64
 5   Cabin         891 non-null    object 
 6   Deck          891 non-null    object 
 7   Embarked_C    891 non-null    bool   
 8   Embarked_Q    891 non-null    bool   
 9   Embarked_S    891 non-null    bool   
 10  Sex_female    891 non-null    bool   
 11  Sex_male      891 non-null    bool   
 12  Deck_Encoded  891 non-null    int64  
dtypes: bool(5), float64(2), int64(4), object(2)
memory usage: 60.2+ KB
None


In [ ]:
# Fill Age based on the average age of that specific Pclass
X_test["Age"] = X_test["Age"].fillna(
    X_test.groupby("Pclass")["Age"].transform("mean")
)

# Create a new category for missing cabins
X_test['Cabin'] = X_test['Cabin'].fillna('Unknown')

# PRO TIP: Just take the first letter of the Cabin (The Deck)
# Decks A, B, C were for wealthier passengers.
X_test['Deck'] = X_test['Cabin'].str[0]

# This automatically turns all text columns into 1s and 0s
X_test = pd.get_dummies(X_test, columns=['Embarked', 'Sex'])

# Method A: Manual Mapping (Best for complete control)
deck_mapping = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7, "T": 8, "U": 8}
X_test['Deck_Encoded'] = X_test['Deck'].map(deck_mapping)

'''
One Tiny "Gotcha" to Check
There is one small thing that might trip you up later: The "Unknown" Deck.

When you ran X_test_dataset['Cabin'].str[0] on the values you filled with "Unknown",
your new Deck column now contains the letter 'U' (the first letter of "Unknown").

This is actually good! It gives your model a specific category for "People with 
no recorded cabin."

Just be careful: Ensure your final model treats 'U' as a category and not as an 
actual deck near 'T' or 'V'.
'''
print(X_test.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Pclass        418 non-null    int64  
 1   Age           418 non-null    float64
 2   SibSp         418 non-null    int64  
 3   Parch         418 non-null    int64  
 4   Fare          417 non-null    float64
 5   Cabin         418 non-null    object 
 6   Deck          418 non-null    object 
 7   Embarked_C    418 non-null    bool   
 8   Embarked_Q    418 non-null    bool   
 9   Embarked_S    418 non-null    bool   
 10  Sex_female    418 non-null    bool   
 11  Sex_male      418 non-null    bool   
 12  Deck_Encoded  418 non-null    int64  
dtypes: bool(5), float64(2), int64(4), object(2)
memory usage: 28.3+ KB
None


In [ ]:
# Visualizing datasets

import matplotlib.pyplot as plt
import seaborn as sns

numeric_cols = X_train.select_dtypes(include=np.number).columns
n_cols = 3
n_rows = int(np.ceil(len(numeric_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.histplot(X_train[col].dropna(), kde=True, ax=axes[i], bins=30)
    axes[i].set_title(col)

for ax in axes[len(numeric_cols):]:
    ax.set_visible(False)

plt.tight_layout()
# Calculate survival rate by Pclass
survival_by_pclass = X_train.groupby('Pclass')['Survived'].mean()

# Create bar chart
sns.barplot(x=survival_by_pclass.index, y=survival_by_pclass.values)
plt.title('Survival Rate by Pclass')
plt.ylabel('Survival Rate')
plt.show()

In [ ]:
# Creating Logistic Regression model

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix